# Practice: features and leakage
*AI from scratch, stage 3* · lesson: https://ihor.sh/ai/features-and-leakage.html

The lesson's bank calls, all 41,188 of them: find a sentinel code, catch a leaking column, build a pipeline that
learns only from training rows, handle a rare "yes", and see how much a random split flatters a model when the
world changes over time. Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it;
`hint(n)` and `answer(n)` are there when you're stuck. On Kaggle, turn on *Internet* in the notebook settings.

Data: Bank Marketing (Moro, Cortez and Rita, UCI Machine Learning Repository, CC BY 4.0).

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['A comparison makes a True/False column; its .mean() is the share that are True.', 'share_yes = (df["y"] == "yes").mean()'], 2: ['!= means "is not equal"; a comparison on a column gives one True/False per row.', 'df["contacted_before"] = df["pdays"] != 999'], 3: ["It's the strongest single column. What does it measure, and when is that number known?", "The call's length is only known when the call is over, and by then you know the answer."], 4: ['make_column_transformer((step, columns), (step, columns)): one pair per group of columns.', 'prep = make_column_transformer((OneHotEncoder(handle_unknown="ignore"), cats), (StandardScaler(), nums))'], 5: ['make_pipeline(prep, LogisticRegression(max_iter=2000, class_weight="balanced")).fit(Xtr, ytr)', 'recall_weighted = recall_score(yte, weighted.predict(Xte))'], 6: ['cut = int(len(X) * 0.75); X.iloc[:cut] is the past, X.iloc[cut:] the future.', 'Xtr_t, Xte_t, ytr_t, yte_t = X.iloc[:cut], X.iloc[cut:], y.iloc[:cut], y.iloc[cut:]'], 7: ['make_pipeline(step1, step2) chains the selection and the model, so each fold selects on its own training rows.', 'inside = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression()), then cross_val_score(inside, Xn, yn, cv=5).mean()']}
_ANSWERS = {1: 'c2hhcmVfeWVzID0gKGRmWyJ5Il0gPT0gInllcyIpLm1lYW4oKQ==', 2: 'ZGZbImNvbnRhY3RlZF9iZWZvcmUiXSA9IGRmWyJwZGF5cyJdICE9IDk5OQ==', 3: 'bGVhayA9ICJkdXJhdGlvbiI=', 4: 'cHJlcCA9IG1ha2VfY29sdW1uX3RyYW5zZm9ybWVyKChPbmVIb3RFbmNvZGVyKGhhbmRsZV91bmtub3duPSJpZ25vcmUiKSwgY2F0cyksIChTdGFuZGFyZFNjYWxlcigpLCBudW1zKSk=', 5: 'd2VpZ2h0ZWQgPSBtYWtlX3BpcGVsaW5lKHByZXAsIExvZ2lzdGljUmVncmVzc2lvbihtYXhfaXRlcj0yMDAwLCBjbGFzc193ZWlnaHQ9ImJhbGFuY2VkIikpLmZpdChYdHIsIHl0cikKcmVjYWxsX3dlaWdodGVkID0gcmVjYWxsX3Njb3JlKHl0ZSwgd2VpZ2h0ZWQucHJlZGljdChYdGUpKQ==', 6: 'Y3V0ID0gaW50KGxlbihYKSAqIDAuNzUpClh0cl90LCBYdGVfdCwgeXRyX3QsIHl0ZV90ID0gWC5pbG9jWzpjdXRdLCBYLmlsb2NbY3V0Ol0sIHkuaWxvY1s6Y3V0XSwgeS5pbG9jW2N1dDpd', 7: 'aW5zaWRlID0gbWFrZV9waXBlbGluZShTZWxlY3RLQmVzdChmX2NsYXNzaWYsIGs9MjApLCBMb2dpc3RpY1JlZ3Jlc3Npb24oKSkKaG9uZXN0X3Njb3JlID0gY3Jvc3NfdmFsX3Njb3JlKGluc2lkZSwgWG4sIHluLCBjdj01KS5tZWFuKCk='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import io, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

raw = urllib.request.urlopen("https://archive.ics.uci.edu/static/public/222/bank+marketing.zip").read()
with zipfile.ZipFile(io.BytesIO(raw)) as outer:                       # a zip inside a zip
    with zipfile.ZipFile(io.BytesIO(outer.read("bank-additional.zip"))) as inner:
        df = pd.read_csv(inner.open("bank-additional/bank-additional-full.csv"), sep=";")
print(df.shape, "calls, in date order from May 2008 to November 2010")
df.head()

## 1. Look first
Set `share_yes`: the fraction of calls that ended in a yes (the `y` column says `"yes"` or `"no"`).

In [ ]:
share_yes = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print(share_yes)

In [ ]:
check(1, lambda: abs(share_yes - 0.1127) < 0.001, "about 11.3% said yes, so saying 'no' to everyone is already 88.7% accurate")

## 2. A code that pretends to be a number
`pdays` is the days since the person was last contacted, except that 999 means "never contacted". Add a column
`contacted_before`: True where `pdays` is not 999. The cell then compares the yes rates.

In [ ]:
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
if "contacted_before" in df:
    print(df.groupby("contacted_before")["y"].apply(lambda s: (s == "yes").mean()))

In [ ]:
check(2, lambda: df["contacted_before"].sum() == 1515, "1,515 people were contacted before (pdays is not 999), and about 64% of them said yes")

## 3. Find the leak
A quick screen for leaks: how well does each number column predict the answer **on its own**? The cell computes
each column's AUC (chapter 12), as if the column's value were the model's score. Look at the top of the list and
set `leak` to the name of the column that can't be known before the call is made. (The next three, employment,
the interest rate and its change, aren't leaks: they're known in advance. But they drift slowly over the years,
which matters in part 6.)

In [ ]:
from sklearn.metrics import roc_auc_score
y = (df["y"] == "yes").astype(int)
numbers = df.drop(columns=["y"]).select_dtypes("number").columns
alone = {c: max(roc_auc_score(y, df[c]), 1 - roc_auc_score(y, df[c])) for c in numbers}   # either direction
print(pd.Series(alone).sort_values(ascending=False).round(3))

In [ ]:
leak = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...

In [ ]:
check(3, lambda: leak == "duration", "duration (how long the call lasted) is known only after the call, when the answer is known too")

## 4. A pipeline that learns from training rows only
The honest columns are everything except the answer and the leak. Split them 75/25 with `stratify=y` (done for you),
then build `prep` with `make_column_transformer`, as in the lesson's imbalance box: one-hot encoding
(`OneHotEncoder(handle_unknown="ignore")`) for the category columns `cats`, `StandardScaler()` for the number
columns `nums`. The cell puts it in a pipeline with logistic regression and measures the AUC.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score

X = df.drop(columns=["y", "duration", "contacted_before"], errors="ignore")
cats = list(X.select_dtypes(exclude="number").columns)
nums = [c for c in X.columns if c not in cats]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print("category columns:", cats)
print("number columns:", nums)
prep = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
if prep is not None:
    model = make_pipeline(prep, LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
    auc_random = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
    recall_plain = recall_score(yte, model.predict(Xte))
    print("AUC", auc_random, " recall of yes at threshold 0.5:", recall_plain)

In [ ]:
check(4, lambda: 0.77 < auc_random < 0.81, "about 0.79 with honest columns on a random split")

The recall is low: at the default threshold, the model finds only about a fifth of the people who'd say yes.

## 5. A rare yes
Fit `weighted`: the same pipeline with `LogisticRegression(max_iter=2000, class_weight="balanced")`, on `Xtr, ytr`.
Then set `recall_weighted`: its recall on the test rows (`recall_score(yte, weighted.predict(Xte))`).

In [ ]:
weighted = recall_weighted = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
if weighted is not None:
    print("recall", recall_plain, "→", recall_weighted, "  precision", precision_score(yte, weighted.predict(Xte)),
          "  AUC", roc_auc_score(yte, weighted.predict_proba(Xte)[:, 1]))

In [ ]:
check(5, lambda: recall_weighted > 0.55 and recall_weighted > 2 * recall_plain,
      "balanced class weights should find well over half the yeses (about 0.62), roughly three times as many")

Precision fell (more calls flagged, a smaller share of them yes) and the AUC hardly moved: weighting moves the
threshold, not the ranking.

## 6. Split by time
The rows are in date order. Train on the first 75% of calls and test on the last 25%, the way the model would
really be used. Set `cut` to the row where the test period starts, then `Xtr_t, Xte_t, ytr_t, yte_t` with `.iloc`
(rows before `cut` and from `cut` on). The cell refits the same pipeline and measures it.

In [ ]:
cut = Xtr_t = Xte_t = ytr_t = yte_t = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...
if Xtr_t is not None:
    timed = make_pipeline(prep, LogisticRegression(max_iter=2000)).fit(Xtr_t, ytr_t)
    auc_time = roc_auc_score(yte_t, timed.predict_proba(Xte_t)[:, 1])
    print("random split AUC", auc_random, "  time split AUC", auc_time)
    print("share yes in the training period:", ytr_t.mean(), "  in the test period:", yte_t.mean())

In [ ]:
check(6, lambda: cut == 30891 and auc_time < auc_random - 0.05,
      "the time split should start at row 30,891 and score clearly lower: the later months behave differently")

In [ ]:
roll = y.rolling(1000, center=True).mean()
plt.figure(figsize=(9, 3))
plt.plot(roll.values, label="share saying yes (1,000-call window)")
plt.plot(df["euribor3m"].values / 10, label="interest rate ÷ 10")
plt.axvline(cut if cut else int(len(df) * 0.75), color="r", ls="--", label="time split")
plt.xlabel("call number (time order)"); plt.legend(); plt.show()

The training period mostly comes from before the crash, when few people said yes; the test period from after,
when many did. A random split mixes both periods into training and hides that. On launch day, the model always
meets the future.

## 7. A step that saw the test rows
The lesson's noise demo: 100 people, 5,000 random columns, coin-flip labels. The first cell chooses 20 columns
with all the labels in view, then cross-validates: it looks like real skill. Set `inside` to a pipeline with
`SelectKBest(f_classif, k=20)` and `LogisticRegression()`, and `honest_score` to its 5-fold
`cross_val_score(...).mean()` on the full `Xn, yn`.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score
rng = np.random.default_rng(0)
Xn, yn = rng.standard_normal((100, 5000)), rng.integers(0, 2, 100)
chosen = SelectKBest(f_classif, k=20).fit_transform(Xn, yn)
print("select first, then cross-validate:", cross_val_score(LogisticRegression(), chosen, yn, cv=5).mean())

In [ ]:
inside = honest_score = None
# YOUR CODE HERE (stuck? hint(7), then answer(7))
...
print("selection inside the pipeline:", honest_score)

In [ ]:
check(7, lambda: honest_score < 0.65, "on coin flips the honest score is about 0.5: there is nothing to learn")

## Next
Back to the path: https://ihor.sh/ai/ · Kaggle Learn's "Intermediate Machine Learning" course has short lessons
on pipelines and data leakage: https://www.kaggle.com/learn/intermediate-machine-learning